<a href="https://colab.research.google.com/github/CarlosRea/Homework-1-Intro-and-Data-Sources/blob/main/Homework_1_Intro_and_Data_Sources.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import io
import requests
import pandas as pd

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
response = requests.get(url, headers=headers)

# Extraer primera tabla de componentes
df_sp500 = pd.read_html(io.StringIO(response.text))[0]

# Extraer el año de la columna 'Date added'
df_sp500['Year_Added'] = pd.to_datetime(df_sp500['Date added'], errors='coerce').dt.year

# Conteo por año desde 2020
additions = df_sp500[df_sp500['Year_Added'] >= 2020]['Year_Added'].value_counts().sort_index()
print("Adiciones por año desde 2020:\n", additions)
print("\nAño con mayor número de adiciones:", int(additions.idxmax()))

# Pregunta adicional: empresas con más de 20 años en el índice
more_than_20 = (df_sp500['Year_Added'] < (2026 - 20)).sum()
print(f"Empresas en el índice > 20 años: {more_than_20}")

Adiciones por año desde 2020:
 Year_Added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
Name: count, dtype: int64

Año con mayor número de adiciones: 2025
Empresas en el índice > 20 años: 218


In [5]:
import warnings
import yfinance as yf
import pandas as pd

warnings.filterwarnings('ignore')

indexes = {
    'US (S&P 500)': '^GSPC',
    'China (Shanghai)': '000001.SS',
    'Hong Kong (Hang Seng)': '^HSI',
    'Australia (ASX 200)': '^AXJO',
    'India (Nifty 50)': '^NSEI',
    'Canada (TSX)': '^GSPTSE',
    'Germany (DAX)': '^GDAXI',
    'United Kingdom (FTSE)': '^FTSE',
    'Japan (Nikkei 225)': '^N225',
    'Mexico (IPC)': '^MXX',
    'Brazil (Ibovespa)': '^BVSP'
}

returns = {}
for name, ticker in indexes.items():
    data = yf.download(ticker, start='2026-01-01', end='2026-08-22', progress=False)
    close = data['Close'].dropna()
    if len(close) >= 2:
        p_start = float(close.iloc[0])
        p_end = float(close.iloc[-1])
        returns[name] = (p_end / p_start - 1) * 100

sp500_ret = returns['US (S&P 500)']
print(f"Rendimiento S&P 500 YTD: {sp500_ret:.2f}%\n")

better_count = 0
for name, ret in returns.items():
    if name != 'US (S&P 500)':
        better = ret > sp500_ret
        if better:
            better_count += 1
        print(f"{name:25}: {ret:6.2f}% | ¿Supera al S&P?: {better}")

print(f"\nTotal de índices que superan al S&P 500: {better_count}")

Rendimiento S&P 500 YTD: 11.90%

China (Shanghai)         :  -2.94% | ¿Supera al S&P?: False
Hong Kong (Hang Seng)    :  -1.25% | ¿Supera al S&P?: False
Australia (ASX 200)      :   3.79% | ¿Supera al S&P?: False
India (Nifty 50)         :  -7.25% | ¿Supera al S&P?: False
Canada (TSX)             :  14.86% | ¿Supera al S&P?: True
Germany (DAX)            :   6.51% | ¿Supera al S&P?: False
United Kingdom (FTSE)    :   8.70% | ¿Supera al S&P?: False
Japan (Nikkei 225)       :  27.36% | ¿Supera al S&P?: True
Mexico (IPC)             :   2.48% | ¿Supera al S&P?: False
Brazil (Ibovespa)        :   6.54% | ¿Supera al S&P?: False

Total de índices que superan al S&P 500: 2


In [6]:
import yfinance as yf
import pandas as pd
import numpy as np

# Descargar histórico diario desde 1950
sp = yf.download('^GSPC', start='1950-01-01', progress=False)
prices = sp['Close'].dropna()
if isinstance(prices, pd.DataFrame):
    prices = prices.iloc[:, 0]

# Identificar máximos históricos (All-Time Highs)
cummax = prices.cummax()
ath_indices = np.where(prices == cummax)[0]

corrections = []
for i in range(len(ath_indices) - 1):
    start_idx = ath_indices[i]
    end_idx = ath_indices[i + 1]

    if end_idx - start_idx > 1:
        high = prices.iloc[start_idx]
        in_between = prices.iloc[start_idx + 1:end_idx]
        low = in_between.min()

        dd = (high - low) / high * 100
        duration = (prices.index[end_idx] - prices.index[start_idx]).days

        if dd >= 5.0:
            corrections.append({'drawdown': dd, 'duration': duration})

df_corr = pd.DataFrame(corrections)
median_dd = df_corr['drawdown'].median()

print(f"Correcciones detectadas: {len(df_corr)}")
print(f"Drawdown Mediana exacta: {median_dd:.2f}%")
print(f"Percentil 25: {df_corr['drawdown'].quantile(0.25):.2f}%")
print(f"Percentil 75: {df_corr['drawdown'].quantile(0.75):.2f}%")

Correcciones detectadas: 74
Drawdown Mediana exacta: 7.99%
Percentil 25: 6.23%
Percentil 75: 14.02%


In [7]:
import yfinance as yf
import pandas as pd

amzn = yf.Ticker('AMZN')
earnings = amzn.get_earnings_dates()

# Filtrar sorpresas positivas válidas
earnings = earnings.dropna(subset=['Surprise(%)'])
pos_surprises = earnings[earnings['Surprise(%)'] > 0].copy()

# Descargar histórico de precios
prices = amzn.history(start='2020-01-01')
prices.index = pd.to_datetime(prices.index).tz_localize(None).normalize()
pos_surprises.index = pd.to_datetime(pos_surprises.index).tz_localize(None).normalize()

two_day_returns = []
for edate in pos_surprises.index:
    # Buscar índice del día de earnings o el día de negociación inmediato
    trading_days = prices.index[prices.index >= edate]
    if len(trading_days) == 0:
        continue
    idx = prices.index.get_loc(trading_days[0])

    # Day 1 = idx - 1, Day 3 = idx + 1
    if 0 < idx < len(prices) - 1:
        p_day1 = prices['Close'].iloc[idx - 1]
        p_day3 = prices['Close'].iloc[idx + 1]
        ret_2d = (p_day3 / p_day1 - 1) * 100
        two_day_returns.append(ret_2d)

s_ret = pd.Series(two_day_returns)
print(f"Mediana del retorno a 2 días: {s_ret.median():.2f}%")

Mediana del retorno a 2 días: 0.35%
